# RAG Pipeline - Exp7

* About
  * Able to run RAG pipelines in parallel as Exp4, but using local models and Railway saved embeddings
  * Compare the eval results with Exp4's eval results

In [1]:
%load_ext autoreload
%autoreload 2

import os
import yaml
os.environ.pop("OPENAI_API_KEY", None)

from datasets import load_dataset
from llama_index.core import Document
from sqlalchemy import make_url
import pandas as pd

from utils_railway import *

import warnings
warnings.filterwarnings('ignore')

resource module not available on Windows
Connected to database: railway


### Load Data Input

In [3]:
fiqa_eval = load_dataset("explodinggradients/fiqa", "ragas_eval")['baseline']

output_dir = "fiqa_raw_text"
os.makedirs(output_dir, exist_ok=True)

rag_lst = []
documents = []  # to store documents for llamindex
for idx, record in enumerate(fiqa_eval):
    context = ''.join(record['contexts'])
    gt = ''.join(record['ground_truths'])
    if 'answer' in record.keys():
        ai0_answer = record['answer'].strip()
    else:
        ai0_answer = None

    with open(os.path.join(output_dir, f"{idx}.txt"), "w", encoding="utf-8") as f:
        f.write(context)
    rag_lst.append({
        'question': record['question'],
        'context': context,
        'context_ct': len(record['contexts']),
        'ground_truth': gt,
        'ai0_answer': ai0_answer
    })
    doc = Document(
        text=context,
        metadata={
            "doc_name": idx
        }
    )
    documents.append(doc)

rag_df = pd.DataFrame(rag_lst)
print(rag_df.shape, max(rag_df['context_ct']), min(rag_df['context_ct']))
print(len(documents))
rag_df.head()

(30, 5) 1 1
30


,question,context,context_ct,ground_truth,ai0_answer
0,How to deposit a cheque issued to an associate...,Just have the associate sign the back and then...,1,Have the check reissued to the proper payee.Ju...,The best way to deposit a cheque issued to an ...
1,Can I send a money order from USPS as a business?,Sure you can. You can fill in whatever you wa...,1,Sure you can. You can fill in whatever you wa...,"Yes, you can send a money order from USPS as a..."
2,1 EIN doing business under multiple business n...,You're confusing a lot of things here. Company...,1,You're confusing a lot of things here. Company...,"Yes, it is possible to have one EIN doing busi..."
3,Applying for and receiving business credit,Set up a meeting with the bank that handles yo...,1,"""I'm afraid the great myth of limited liabilit...",Applying for and receiving business credit can...
4,401k Transfer After Business Closure,The time horizon for your 401K/IRA is essentia...,1,You should probably consult an attorney. Howev...,If your employer has closed and you need to tr...


In [4]:
with open("all_rag_pipeline_config.yaml", "r", encoding="utf-8") as f:
     all_config= yaml.safe_load(f)

In [10]:
cfgs = []
for i in range(len(all_config['yaml_config_files'])):
    yaml_config_file = 'output/saved_configs/' + all_config['yaml_config_files'][i]
    cfgs.append(yaml_config_file)
    config = {
        'llm_model': all_config['llm_models'][i],
        'embedding_model': all_config['embedding_models'][i],
        'output_file': all_config['output_files'][i],
        'retriever_params': all_config['retriever_settings'][i]['retriever_params'],
        'embedding_model_settings': all_config['embedding_model_settings'][0],
    }

    with open(yaml_config_file, "w", encoding="utf-8") as f:
        yaml.safe_dump(config, f, sort_keys=False)

In [11]:
DATABASE_URL_PUBLIC = os.getenv("DATABASE_URL_PUBLIC_RAG_DR")
url = make_url(DATABASE_URL_PUBLIC)

max_worker = os.cpu_count() - 1
await run_all_in_processes(cfgs, rag_lst, documents, url, max_workers=max_worker)

### Evaluation

In [5]:
from langchain_groq import ChatGroq
import nest_asyncio
nest_asyncio.apply()

with open('eval_prompts.yaml', 'r') as file:
    prompt_versions = yaml.safe_load(file)

eval_llm = ChatGroq(
    groq_api_key=os.environ["GROQ_TOKEN"],
    model_name="openai/gpt-oss-20b", 
    temperature=0.7
)

eval_input_lst = []
for config in all_config['yaml_config_files']:
    yaml_config_file = 'output/saved_configs/' + config
    with open(yaml_config_file, "r", encoding="utf-8") as f:
        config = yaml.safe_load(f)
    output_file = config['output_file']
    response_lst = json.load(open(output_file, "r", encoding="utf-8"))
    eval_input = get_eval_input(response_lst)
    eval_input = pd.merge(eval_input, rag_df[['question', 'context']], left_on='query', right_on='question')
    eval_input.drop(columns=['question'], inplace=True)
    eval_input_lst.append(eval_input)
    print(eval_input.shape)
    display(eval_input.head())

(30, 5)


,query,ai_answer,referenced_answer,retrieved_content,context
0,How to deposit a cheque issued to an associate...,To deposit a cheque issued to an associate in ...,Have the check reissued to the proper payee.Ju...,Just have the associate sign the back and then...,Just have the associate sign the back and then...
1,Can I send a money order from USPS as a business?,"Yes, you can fill in your business name and ad...",Sure you can. You can fill in whatever you wa...,Sure you can. You can fill in whatever you wa...,Sure you can. You can fill in whatever you wa...
2,1 EIN doing business under multiple business n...,You can have one Employer Identification Numbe...,You're confusing a lot of things here. Company...,Sure you can. You can fill in whatever you wa...,You're confusing a lot of things here. Company...
3,Applying for and receiving business credit,Establishing a relationship with a banker is c...,"""I'm afraid the great myth of limited liabilit...",Set up a meeting with the bank that handles yo...,Set up a meeting with the bank that handles yo...
4,401k Transfer After Business Closure,If your employer's 401(k) plan is terminated d...,You should probably consult an attorney. Howev...,The time horizon for your 401K/IRA is essentia...,The time horizon for your 401K/IRA is essentia...


(30, 5)


,query,ai_answer,referenced_answer,retrieved_content,context
0,How to deposit a cheque issued to an associate...,1. **Endorse the check** – Have the associate ...,Have the check reissued to the proper payee.Ju...,Just have the associate sign the back and then...,Just have the associate sign the back and then...
1,Can I send a money order from USPS as a business?,Yes. The USPS money order lets you put any nam...,Sure you can. You can fill in whatever you wa...,Sure you can. You can fill in whatever you wa...,Sure you can. You can fill in whatever you wa...
2,1 EIN doing business under multiple business n...,Yes. A single Employer Identification Number c...,You're confusing a lot of things here. Company...,Sure you can. You can fill in whatever you wa...,You're confusing a lot of things here. Company...
3,Applying for and receiving business credit,"To obtain business credit, follow these steps:...","""I'm afraid the great myth of limited liabilit...",Set up a meeting with the bank that handles yo...,Set up a meeting with the bank that handles yo...
4,401k Transfer After Business Closure,"When your employer shuts down, the plan will u...",You should probably consult an attorney. Howev...,The time horizon for your 401K/IRA is essentia...,The time horizon for your 401K/IRA is essentia...


In [6]:
for idx, eval_input in enumerate(eval_input_lst):
    retrieval_quality = asyncio.run(get_retrieval_quality_output_async(eval_input, eval_llm,
                                                            prompt_versions['rq_prompt_template']))
    retrieval_quality['same_context'] = retrieval_quality['retrieved_content'] == retrieval_quality['context']
    retrieval_quality.to_excel(all_config['eval_rq_output'][idx], index=False)
    print(retrieval_quality['retrieval_quality_score'].value_counts())
    print(retrieval_quality['same_context'].value_counts())
    display(retrieval_quality.head(n=2))

retrieval_quality_score
3    20
0     6
2     3
1     1
Name: count, dtype: int64
same_context
True     19
False    11
Name: count, dtype: int64


,query,ai_answer,referenced_answer,retrieved_content,context,retrieval_quality_score,rq_reasoning,same_context
0,How to deposit a cheque issued to an associate...,To deposit a cheque issued to an associate in ...,Have the check reissued to the proper payee.Ju...,Just have the associate sign the back and then...,Just have the associate sign the back and then...,3,The retrieved content mirrors the context exac...,True
1,Can I send a money order from USPS as a business?,"Yes, you can fill in your business name and ad...",Sure you can. You can fill in whatever you wa...,Sure you can. You can fill in whatever you wa...,Sure you can. You can fill in whatever you wa...,3,The retrieved content matches the context exac...,True


retrieval_quality_score
3    20
0     6
2     3
1     1
Name: count, dtype: int64
same_context
True     19
False    11
Name: count, dtype: int64


,query,ai_answer,referenced_answer,retrieved_content,context,retrieval_quality_score,rq_reasoning,same_context
0,How to deposit a cheque issued to an associate...,1. **Endorse the check** – Have the associate ...,Have the check reissued to the proper payee.Ju...,Just have the associate sign the back and then...,Just have the associate sign the back and then...,3,The retrieved content repeats the context verb...,True
1,Can I send a money order from USPS as a business?,Yes. The USPS money order lets you put any nam...,Sure you can. You can fill in whatever you wa...,Sure you can. You can fill in whatever you wa...,Sure you can. You can fill in whatever you wa...,3,The retrieved content directly answers the use...,True


In [9]:
for idx, eval_input in enumerate(eval_input_lst):
    answer_quality = asyncio.run(get_answer_quality_output_async(eval_input, eval_llm,
                                                            prompt_versions['aq_prompt_template']))
    answer_quality.to_excel(all_config['eval_aq_output'][idx], index=False)
    print(answer_quality.shape)
    print(answer_quality['answer_quality_score'].value_counts(normalize=True))
    display(answer_quality.head(n=2))

(30, 7)
answer_quality_score
 2    0.700000
-1    0.133333
 1    0.133333
 3    0.033333
Name: proportion, dtype: float64


,query,ai_answer,referenced_answer,retrieved_content,context,answer_quality_score,aq_reasoning
0,How to deposit a cheque issued to an associate...,To deposit a cheque issued to an associate in ...,Have the check reissued to the proper payee.Ju...,Just have the associate sign the back and then...,Just have the associate sign the back and then...,2,The AI’s answer correctly states that the asso...
1,Can I send a money order from USPS as a business?,"Yes, you can fill in your business name and ad...",Sure you can. You can fill in whatever you wa...,Sure you can. You can fill in whatever you wa...,Sure you can. You can fill in whatever you wa...,2,The AI’s answer correctly states that a busine...


(30, 7)
answer_quality_score
 4    0.366667
 2    0.266667
 3    0.233333
-1    0.100000
 1    0.033333
Name: proportion, dtype: float64


,query,ai_answer,referenced_answer,retrieved_content,context,answer_quality_score,aq_reasoning
0,How to deposit a cheque issued to an associate...,1. **Endorse the check** – Have the associate ...,Have the check reissued to the proper payee.Ju...,Just have the associate sign the back and then...,Just have the associate sign the back and then...,4,The AI’s answer addresses the user’s question ...
1,Can I send a money order from USPS as a business?,Yes. The USPS money order lets you put any nam...,Sure you can. You can fill in whatever you wa...,Sure you can. You can fill in whatever you wa...,Sure you can. You can fill in whatever you wa...,3,Both answers confirm that USPS money orders ca...


In [10]:
for idx in range(len(eval_input_lst)):
    retrieval_quality = pd.read_excel(all_config['eval_rq_output'][idx])
    answer_quality = pd.read_excel(all_config['eval_aq_output'][idx])
    all_df = retrieval_quality.merge(answer_quality[['query', 'answer_quality_score', 'aq_reasoning']], on='query')
    all_df.to_excel(all_config['eval_all_output'][idx], index=False)
    print(all_df.shape)
    display(all_df.head(n=2))

(30, 10)


,query,ai_answer,referenced_answer,retrieved_content,context,retrieval_quality_score,rq_reasoning,same_context,answer_quality_score,aq_reasoning
0,How to deposit a cheque issued to an associate...,To deposit a cheque issued to an associate in ...,Have the check reissued to the proper payee.Ju...,Just have the associate sign the back and then...,Just have the associate sign the back and then...,3,The retrieved content mirrors the context exac...,True,2,The AI’s answer correctly states that the asso...
1,Can I send a money order from USPS as a business?,"Yes, you can fill in your business name and ad...",Sure you can. You can fill in whatever you wa...,Sure you can. You can fill in whatever you wa...,Sure you can. You can fill in whatever you wa...,3,The retrieved content matches the context exac...,True,2,The AI’s answer correctly states that a busine...


(30, 10)


,query,ai_answer,referenced_answer,retrieved_content,context,retrieval_quality_score,rq_reasoning,same_context,answer_quality_score,aq_reasoning
0,How to deposit a cheque issued to an associate...,1. **Endorse the check** – Have the associate ...,Have the check reissued to the proper payee.Ju...,Just have the associate sign the back and then...,Just have the associate sign the back and then...,3,The retrieved content repeats the context verb...,True,4,The AI’s answer addresses the user’s question ...
1,Can I send a money order from USPS as a business?,Yes. The USPS money order lets you put any nam...,Sure you can. You can fill in whatever you wa...,Sure you can. You can fill in whatever you wa...,Sure you can. You can fill in whatever you wa...,3,The retrieved content directly answers the use...,True,3,Both answers confirm that USPS money orders ca...
